<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/advance_pyspark/Repartition_and_Coalesce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark Training').getOrCreate()
sc = spark.sparkContext

In [3]:
print(os.path.exists("/content/employees.csv"))

True


In [4]:
emp_df=spark.read.csv("/content/employees.csv",header=True,inferSchema=True)
emp_df.printSchema()

root
 |-- Employee ID: integer (nullable = true)
 |-- Store ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Position: string (nullable = true)



In [5]:
emp_df.count()

404

In [6]:
emp_df.rdd.getNumPartitions()

1

In [8]:
repartioned_df=emp_df.repartition(4)
repartioned_df.rdd.getNumPartitions()

4

In [10]:
from pyspark.sql.functions import*
repartioned_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()

+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|  101|
|          1|  101|
|          2|  101|
|          3|  101|
+-----------+-----+



In [11]:
repartition_with_column_df=emp_df.repartition(3,col("Name"))
repartition_with_column_df.rdd.getNumPartitions()

3

In [12]:
repartition_with_column_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()

+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|  133|
|          1|  137|
|          2|  134|
+-----------+-----+



In [14]:
repartition_exceed_df=emp_df.repartition(300,col("Name"))
repartition_exceed_df.rdd.getNumPartitions()

300

In [15]:
repartition_exceed_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()

+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|    2|
|          1|    1|
|          2|    2|
|          4|    3|
|          5|    2|
|          6|    1|
|          8|    2|
|          9|    3|
|         10|    1|
|         14|    1|
|         15|    1|
|         16|    1|
|         18|    2|
|         19|    2|
|         21|    1|
|         22|    2|
|         23|    4|
|         24|    1|
|         25|    3|
|         26|    2|
+-----------+-----+
only showing top 20 rows



In [16]:
coalesced_df=emp_df.coalesce(4)
coalesced_df.rdd.getNumPartitions()

1

In [17]:
repartioned_df1=emp_df.repartition(8)
repartioned_df1.rdd.getNumPartitions()

8

In [18]:
coalesced_df=repartioned_df1.coalesce(3)
coalesced_df.rdd.getNumPartitions()

3

In [19]:
coalesced_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()

+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|  101|
|          1|  151|
|          2|  152|
+-----------+-----+



In [20]:
r_df=repartioned_df1.repartition(3)
print(r_df.rdd.getNumPartitions())
r_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()


3
+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|  135|
|          1|  135|
|          2|  134|
+-----------+-----+



In [30]:
spark.conf.get("spark.sql.shuffle.partitions")

'300'

because repartition by default takes in the value present in spark.sql.shuffle.partitions if integer value is not explicitly provided

In [31]:
r2_df=emp_df.repartition("Name")
r2_df.rdd.getNumPartitions()

1

In [32]:
spark.conf.set("spark.sql.shuffle.partitions", "300")
r2_df=emp_df.repartition("Name")
r2_df.rdd.getNumPartitions()

1